# Microsoft Agent Framework — Azure OpenAI (Responses API)

Sa sample ng code na ito, gagamitin mo ang **Microsoft Agent Framework (MAF)** upang gumawa ng isang simpleng ahente na suportado ng **Azure OpenAI** gamit ang **Responses API**.

> **Tandaan sa migrasyon:** Ang sample na ito dati ay gumagamit ng Semantic Kernel kasama ang GitHub Models. Ito ay inilipat na sa Microsoft Agent Framework, at ang GitHub Models (deprecated, magre-retire sa Hulyo 2026) ay pinalitan ng Azure OpenAI, na sumusuporta sa Responses API. Ang `OpenAIChatClient` sa MAF ay naka-target sa stable na `/openai/v1/` endpoint ng Azure OpenAI at ginagamit ang Responses API bilang default.

Ang layunin ng sample na ito ay ipakita ang mga hakbang na ipapatupad sa mga karagdagang sample ng code kapag nag-iimplementa ng iba't ibang mga agentic na pattern.


In [ ]:
%pip install agent-framework agent-framework-openai azure-identity -q


## I-import ang Mga Kinakailangang Package sa Python


In [ ]:
import os
import random

from dotenv import load_dotenv
from IPython.display import display, HTML

from agent_framework import tool
from agent_framework.openai import OpenAIChatClient
from azure.identity import AzureCliCredential


## Pagde-define ng Isang Tool

Sa Microsoft Agent Framework, ang **tool** ay isang simpleng Python function na may dekorasyong `@tool` na maaaring tawagan ng agent. Sa ibaba namin dine-define ang isang tool na nagbabalik ng random na destinasyon ng bakasyon at iniiwasang ulitin ang naunang destinasyon.


In [ ]:
# A list of vacation destinations the tool can choose from.
_DESTINATIONS = [
    "Barcelona, Spain",
    "Paris, France",
    "Berlin, Germany",
    "Tokyo, Japan",
    "Sydney, Australia",
    "New York, USA",
    "Cairo, Egypt",
    "Cape Town, South Africa",
    "Rio de Janeiro, Brazil",
    "Bali, Indonesia",
]

# Track the last destination so repeated calls avoid immediate repeats.
_last_destination: str | None = None


@tool(approval_mode="never_require")
def get_random_destination() -> str:
    """Provides a random vacation destination."""
    global _last_destination
    available = _DESTINATIONS.copy()
    if _last_destination and len(available) > 1:
        available.remove(_last_destination)
    destination = random.choice(available)
    _last_destination = destination
    return destination


In [ ]:
load_dotenv()

endpoint = os.environ["AZURE_OPENAI_ENDPOINT"]
deployment = os.environ["AZURE_OPENAI_DEPLOYMENT"]

# OpenAIChatClient targets Azure OpenAI's v1 endpoint and uses the Responses API.
# Sign in with `az login` first so AzureCliCredential can authenticate.
chat_client = OpenAIChatClient(
    model=deployment,
    azure_endpoint=endpoint,
    credential=AzureCliCredential(),
)


## Paglikha ng Ahente

Dito, gumawa tayo ng Ahente na pinangalanang `TravelAgent`.

Sa halimbawang ito, gumamit tayo ng napakapayak na mga tagubilin. Malayang baguhin ang mga tagubiling ito upang makita kung paano nagbabago ang pag-uugali ng ahente.


In [ ]:
agent = chat_client.as_agent(
    name="TravelAgent",
    instructions="You are a helpful AI Agent that can help plan vacations for customers at random destinations",
    tools=[get_random_destination],
)


## Pagpapatakbo ng Ahente

Ngayon, maaari na nating patakbuhin ang ahente. Gumagawa tayo ng isang `AgentSession` upang maalala ng ahente ang usapan sa bawat pag-ikot, pagkatapos ay magpadala ng dalawang `user_inputs`. Ang una ay humihiling ng isang biyahe; ang pangalawa ay nagsasabi na hindi nagustuhan ng gumagamit ang suhestiyon at humihiling ng isa pa — ginagamit ng ahente ang kasaysayan ng session plus ang `get_random_destination` na tool upang tumugon.

Maaari mong baguhin ang mga mensaheng ito upang makita kung paano naiiba ang reaksyon ng ahente. Ang mga tugon ay **streamed** token-isa-isa.


In [ ]:
user_inputs = [
    "Plan me a day trip.",
    "I don't like that destination. Plan me another vacation.",
]


async def main():
    # A session keeps conversation history across turns.
    session = agent.create_session()

    for user_input in user_inputs:
        html_output = (
            f"<div style='margin-bottom:10px'>"
            f"<div style='font-weight:bold'>User:</div>"
            f"<div style='margin-left:20px'>{user_input}</div></div>"
        )

        full_response: list[str] = []
        # Stream the agent's response token-by-token. The agent will call the
        # get_random_destination tool automatically when it needs a destination.
        async for chunk in agent.run(user_input, session=session, stream=True):
            full_response.append(str(chunk))

        html_output += (
            "<div style='margin-bottom:20px'>"
            f"<div style='font-weight:bold'>TravelAgent:</div>"
            f"<div style='margin-left:20px; white-space:pre-wrap'>{''.join(full_response)}</div></div><hr>"
        )

        display(HTML(html_output))


await main()


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**Pagtatanggi**:
Ang dokumentong ito ay isinalin gamit ang serbisyo ng AI translation na [Co-op Translator](https://github.com/Azure/co-op-translator). Bagama't nagsusumikap kami para sa katumpakan, pakatandaan na ang awtomatikong pagsasalin ay maaaring maglaman ng mga pagkakamali o hindi pagkakatugma. Ang orihinal na dokumento sa orihinal nitong wika ang dapat ituring na pangunahing sanggunian. Para sa mahahalagang impormasyon, inirerekomenda ang propesyonal na pagsasalin ng tao. Hindi kami mananagot sa anumang maling pagkakaintindi o maling interpretasyon na nagmula sa paggamit ng pagsasaling ito.
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
